# TinyThinker one-click Colab training

This notebook is designed for Google Colab with a GPU runtime.

Default behavior when you click **Run all**:
1. clones the TinyThinker repo into `/content/TinyThinker`,
2. installs CUDA-enabled PyTorch and Python dependencies,
3. verifies the GPU,
4. uses the repo's checked-in `input.txt`,
5. builds dataset artifacts,
6. trains TinyThinker into a resumable run directory,
7. prints sample generations, and
8. zips and downloads the trained artifacts.

Optional overrides are exposed as variables in later cells.

In [ ]:
from pathlib import Path
import os

REPO_URL = "https://github.com/rexmhall09/TinyThinker.git"
REPO_DIR = Path("/content/TinyThinker")

if not REPO_DIR.exists():
    !git clone "$REPO_URL" "$REPO_DIR"
else:
    print(f"Using existing repo at {REPO_DIR}")

os.chdir(REPO_DIR)
print(f"Working directory: {Path.cwd()}")

In [ ]:
%pip install -q --upgrade pip
%pip install -q torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 --index-url https://download.pytorch.org/whl/cu121
%pip install -q numpy==1.26.4 tqdm==4.67.1 pytest==8.3.5

In [ ]:
from pathlib import Path

USE_GOOGLE_DRIVE = False
DATA_SOURCE = "repo"  # repo | upload | drive
DRIVE_INPUT_PATH = ""  # set when DATA_SOURCE == "drive"
RUN_NAME = "tinythinker_colab_run"
SAMPLE_PROMPT = "Hi!"

# Right-sized defaults for the included dataset.
N_EMBD = 384
N_HEAD = 6
N_LAYER = 6
BLOCK_SIZE = 256
DROPOUT = 0.1
BATCH_SIZE = 32
MAX_ITERS = 1500
LR = 3e-4
EVAL_INTERVAL = 100
EVAL_STEPS = 20
SAVE_INTERVAL = 100
GRAD_CLIP = 1.0
WEIGHT_DECAY = 0.01
SEED = 42

ARTIFACT_ROOT = Path("/content/tinythinker_artifacts")
DATA_DIR = ARTIFACT_ROOT / "data"
RUN_DIR = ARTIFACT_ROOT / RUN_NAME
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)
RUN_DIR.mkdir(parents=True, exist_ok=True)

print({
    "repo_dir": str(REPO_DIR),
    "data_dir": str(DATA_DIR),
    "run_dir": str(RUN_DIR),
    "data_source": DATA_SOURCE,
    "use_google_drive": USE_GOOGLE_DRIVE,
})

In [ ]:
import subprocess
import torch

print(subprocess.run(["nvidia-smi"], check=False, text=True, capture_output=True).stdout)
print("torch", torch.__version__)
print("torch cuda", torch.version.cuda)
print("cuda available", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("No CUDA device is available. In Colab, switch to a GPU runtime before running all cells.")
print("gpu", torch.cuda.get_device_name(0))

In [ ]:
from pathlib import Path
import shutil

if USE_GOOGLE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")

if DATA_SOURCE == "repo":
    input_path = REPO_DIR / "input.txt"
elif DATA_SOURCE == "upload":
    from google.colab import files

    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("No file uploaded")
    uploaded_name = next(iter(uploaded))
    input_path = Path("/content") / uploaded_name
elif DATA_SOURCE == "drive":
    if not USE_GOOGLE_DRIVE:
        raise RuntimeError("Set USE_GOOGLE_DRIVE = True before using DATA_SOURCE='drive'")
    input_path = Path(DRIVE_INPUT_PATH)
else:
    raise ValueError(f"Unsupported DATA_SOURCE: {DATA_SOURCE}")

if not input_path.exists():
    raise FileNotFoundError(f"Input text not found: {input_path}")

working_input_path = ARTIFACT_ROOT / "input.txt"
shutil.copyfile(input_path, working_input_path)
print(f"Using training data: {working_input_path}")

In [ ]:
import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        "build_memmap.py",
        "--input",
        str(working_input_path),
        "--out-dir",
        str(DATA_DIR),
        "--force",
    ],
    check=True,
)
print(f"Prepared dataset artifacts in {DATA_DIR}")

In [ ]:
import subprocess
import sys

train_command = [
    sys.executable,
    "train.py",
    "--data-dir",
    str(DATA_DIR),
    "--run-dir",
    str(RUN_DIR),
    "--batch-size",
    str(BATCH_SIZE),
    "--max-iters",
    str(MAX_ITERS),
    "--lr",
    str(LR),
    "--eval-interval",
    str(EVAL_INTERVAL),
    "--eval-steps",
    str(EVAL_STEPS),
    "--save-interval",
    str(SAVE_INTERVAL),
    "--grad-clip",
    str(GRAD_CLIP),
    "--weight-decay",
    str(WEIGHT_DECAY),
    "--seed",
    str(SEED),
    "--n-embd",
    str(N_EMBD),
    "--n-head",
    str(N_HEAD),
    "--n-layer",
    str(N_LAYER),
    "--block-size",
    str(BLOCK_SIZE),
    "--dropout",
    str(DROPOUT),
]
subprocess.run(train_command, check=True)
print(f"Training complete. Run artifacts: {RUN_DIR}")

In [ ]:
from prompt import generate_completion, load_model_for_inference

artifact_path = RUN_DIR / "model_final.pt"
model, tokenizer = load_model_for_inference(str(artifact_path), "cuda")
for sample_index in range(3):
    completion = generate_completion(
        model=model,
        tokenizer=tokenizer,
        prompt=SAMPLE_PROMPT,
        device="cuda",
        max_tokens=128,
        temperature=0.8,
        top_k=50,
    )
    print(f"Sample {sample_index + 1}: {completion}")

In [ ]:
import shutil
from pathlib import Path

archive_base = ARTIFACT_ROOT / RUN_NAME
archive_path = Path(shutil.make_archive(str(archive_base), "zip", root_dir=RUN_DIR))
print(f"Created archive: {archive_path}")

if USE_GOOGLE_DRIVE:
    drive_archive_dir = Path("/content/drive/MyDrive/tinythinker_exports")
    drive_archive_dir.mkdir(parents=True, exist_ok=True)
    drive_archive_path = drive_archive_dir / archive_path.name
    shutil.copyfile(archive_path, drive_archive_path)
    print(f"Copied archive to Google Drive: {drive_archive_path}")

from google.colab import files
files.download(str(archive_path))